# Workshop 1 — Source Profiling and Sample Validation

**Team 8 — Tsunami Monitoring** · Data Analysis Programming 2026-II

This notebook reads only the saved samples in `workshop_1/samples/`. Historical queries cover the Pacific basin in 2010–2025; the USGS historical sample covers 2010–2011 with M ≥ 6. The geographic analysis uses `config/pacific_polygon.geojson`. The incremental comparison uses the two saved snapshots from 7 November 2026.

## 1. Setup

In [1]:

import json
import math
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 80)

SAMPLES = Path("../workshop_1/samples")
POLYGON = Path("../config/pacific_polygon.geojson")
FIRST = "20261107_050434"
SECOND = "20261107_140301"
print("Samples folder:", SAMPLES.resolve())
print("Reference snapshots:", FIRST, "and", SECOND)


Samples folder: /home/linmonp/tsunami-monitoring/workshop_1/samples
Reference snapshots: 20261107_050434 and 20261107_140301


## 2. Load Samples

### 2.1 NOAA HazEL — tsunami events (historical)

In [2]:

noaa_files = sorted(f for f in SAMPLES.glob("noaa_tsunamis_pacific*.json") if "2011" not in f.name)
print("Files:", [f.name for f in noaa_files])

noaa = pd.DataFrame(
    [item for f in noaa_files for item in json.loads(f.read_text(encoding="utf-8"))["items"]]
)
print("Rows read:", len(noaa), "| duplicated ids:", noaa["id"].duplicated().sum())

def load_descriptor(name):
    items = json.loads((SAMPLES / f"noaa_descriptor_{name}.json").read_text(encoding="utf-8"))["items"]
    return {int(i["id"]): i["description"] for i in items}

causes = load_descriptor("causes")
validity = load_descriptor("validity")
warning_statuses = load_descriptor("warning_statuses")
regions = load_descriptor("regions")

noaa["cause"] = noaa["causeCode"].map(causes)
noaa["validity"] = noaa["eventValidity"].map(validity)
noaa["warning_status"] = noaa["warningStatusId"].map(warning_statuses)
noaa["region"] = noaa["regionCode"].map(regions)

# HazEL stores the date in six separate columns: build one UTC timestamp.
# Hour/minute/second can be missing; keep a flag instead of inventing values.
noaa["time_complete"] = noaa["hour"].notna() & noaa["minute"].notna()
noaa["time_utc"] = (
    pd.to_datetime(noaa[["year", "month", "day"]], errors="coerce", utc=True)
    + pd.to_timedelta(noaa["hour"].fillna(0), unit="h")
    + pd.to_timedelta(noaa["minute"].fillna(0), unit="m")
    + pd.to_timedelta(noaa["second"].fillna(0), unit="s")
)
print("Unparseable dates:", noaa["time_utc"].isna().sum())
noaa[["id", "time_utc", "time_complete", "latitude", "longitude", "eqMagnitude", "cause", "validity", "region"]].head()


Files: ['noaa_tsunamis_pacific_2010_2025.json', 'noaa_tsunamis_pacific_page2.json']
Rows read: 221 | duplicated ids: 0
Unparseable dates: 0


,id,time_utc,time_complete,latitude,longitude,eqMagnitude,cause,validity,region
0,4522,2010-01-03 21:48:05.300+00:00,True,-8.743,157.477,6.6,Earthquake,Definite Tsunami,"New Caledonia, New Guinea, Solomon Is., Vanuatu"
1,4523,2010-01-03 22:36:27.900+00:00,True,-8.799,157.346,7.1,Earthquake,Definite Tsunami,"New Caledonia, New Guinea, Solomon Is., Vanuatu"
2,4542,2010-01-05 12:15:32.200+00:00,True,-9.019,157.551,6.8,Earthquake,Definite Tsunami,"New Caledonia, New Guinea, Solomon Is., Vanuatu"
3,4682,2010-02-27 06:34:11.500+00:00,True,-36.122,-72.898,8.8,Earthquake,Definite Tsunami,West Coast of South America
4,4722,2010-03-11 14:39:43.900+00:00,True,-34.290,-71.891,6.9,Earthquake,Definite Tsunami,West Coast of South America


### 2.2 USGS FDSN catalog — earthquakes (historical)

In [3]:

usgs_file = SAMPLES / "usgs_earthquakes_2010_2011_m6_pacific.csv"
print("File:", usgs_file.name)

usgs_hist = pd.read_csv(usgs_file)
usgs_hist["time_utc"] = pd.to_datetime(usgs_hist["time"], utc=True)
usgs_hist["updated_utc"] = pd.to_datetime(usgs_hist["updated"], utc=True)
print("Rows:", len(usgs_hist), "| period:", usgs_hist["time_utc"].min(), "->", usgs_hist["time_utc"].max())
usgs_hist.head()


File: usgs_earthquakes_2010_2011_m6_pacific.csv
Rows: 343 | period: 2010-01-02 08:45:33.020000+00:00 -> 2011-12-26 04:48:08.870000+00:00


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,net,id,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource,time_utc,updated_utc
0,2011-12-26T04:48:08.870Z,-16.198,-173.862,90.0,6.0,mww,536,35.4,NaN,0.95,us,usp000jcna,2022-04-08T20:25:06.544Z,"27 km SSW of Hihifo, Tonga",earthquake,NaN,NaN,NaN,NaN,reviewed,us,us,2011-12-26 04:48:08.870+00:00,2022-04-08 20:25:06.544+00:00
1,2011-12-15T10:10:07.210Z,-32.718,-179.099,32.0,6.0,mww,423,21.6,NaN,1.09,us,usp000jc7p,2022-04-08T20:24:26.506Z,south of the Kermadec Islands,earthquake,NaN,NaN,NaN,NaN,reviewed,us,us,2011-12-15 10:10:07.210+00:00,2022-04-08 20:24:26.506+00:00
2,2011-12-14T05:04:58.630Z,-7.551,146.809,135.0,7.1,mww,485,10.6,NaN,0.78,us,usp000jc5z,2022-04-08T20:24:25.265Z,"25 km SSE of Wau, Papua New Guinea",earthquake,NaN,NaN,NaN,NaN,reviewed,us,us,2011-12-14 05:04:58.630+00:00,2022-04-08 20:24:25.265+00:00
3,2011-12-13T07:52:11.930Z,0.041,123.030,161.0,6.0,mww,536,13.9,NaN,0.95,us,usp000jc4z,2022-04-08T20:24:22.823Z,"55 km S of Gorontalo, Indonesia",earthquake,NaN,NaN,NaN,NaN,reviewed,us,us,2011-12-13 07:52:11.930+00:00,2022-04-08 20:24:22.823+00:00
4,2011-12-11T01:47:25.560Z,17.986,-99.789,59.0,6.5,mww,578,38.6,NaN,0.93,us,usp000jc1y,2024-02-23T00:54:36.837Z,"6 km SSW of Nuevo Balsas, Mexico",earthquake,NaN,NaN,NaN,NaN,reviewed,us,us,2011-12-11 01:47:25.560+00:00,2024-02-23 00:54:36.837+00:00


### 2.3 USGS real-time feed — earthquakes (incremental)

In [4]:

rt_file = SAMPLES / f"usgs_realtime_{SECOND}.geojson"
print("File:", rt_file.name)
feed = json.loads(rt_file.read_text(encoding="utf-8"))
print("Feed title:", feed["metadata"]["title"], "| count:", feed["metadata"]["count"])
print("Generated:", pd.to_datetime(feed["metadata"]["generated"], unit="ms", utc=True))

usgs_rt = pd.json_normalize(feed["features"])
usgs_rt.columns = [c.replace("properties.", "") for c in usgs_rt.columns]
# GeoJSON coordinates are [longitude, latitude, depth] (longitude first)
usgs_rt[["lon", "lat", "depth"]] = pd.DataFrame(usgs_rt["geometry.coordinates"].tolist(), index=usgs_rt.index)
usgs_rt = usgs_rt.drop(columns=["geometry.coordinates", "geometry.type", "type"])
usgs_rt["time_utc"] = pd.to_datetime(usgs_rt["time"], unit="ms", utc=True)
usgs_rt["updated_utc"] = pd.to_datetime(usgs_rt["updated"], unit="ms", utc=True)
usgs_rt[["id", "time_utc", "updated_utc", "mag", "magType", "status", "tsunami", "lat", "lon", "depth", "place"]].head()


File: usgs_realtime_20261107_140301.geojson
Feed title: USGS Magnitude 4.5+ Earthquakes, Past Month | count: 726
Generated: 2026-11-07 14:03:01+00:00


,id,time_utc,updated_utc,mag,magType,status,tsunami,lat,lon,depth,place
0,us7000twrt,2026-11-07 11:35:38.338+00:00,2026-11-07 11:55:28.040+00:00,4.9,mb,reviewed,0,-18.2483,-12.9445,10.000,southern Mid-Atlantic Ridge
1,us7000twrq,2026-11-07 11:07:43.355+00:00,2026-11-07 13:39:16.859+00:00,5.5,mww,reviewed,0,36.1884,141.4404,31.748,"74 km NE of Hasaki, Japan"
2,us7000twrp,2026-11-07 10:53:12.299+00:00,2026-11-07 12:46:07.040+00:00,4.5,mb,reviewed,0,-21.6011,-68.6413,110.716,"57 km SW of Ollagüe, Chile"
3,us7000twrh,2026-11-07 09:47:33.240+00:00,2026-11-07 12:35:21.040+00:00,4.7,mb,reviewed,0,0.0075,125.7691,38.531,"156 km ESE of Modisi, Indonesia"
4,us7000twqw,2026-11-07 07:14:10.868+00:00,2026-11-07 07:41:39.040+00:00,4.6,mb,reviewed,0,-57.9579,-25.8023,103.682,South Sandwich Islands region


### 2.4 tsunami.gov — Atom feeds and CAP messages (incremental)

In [5]:

NS = {
    "a": "http://www.w3.org/2005/Atom",
    "geo": "http://www.w3.org/2003/01/geo/wgs84_pos#",
    "cap": "urn:oasis:names:tc:emergency:cap:1.2",
}

def parse_cap(path):
    """Flatten a CAP message into one dictionary."""
    root = ET.parse(path).getroot()
    info = root.find("cap:info", NS)
    row = {
        "identifier": root.findtext("cap:identifier", namespaces=NS),
        "sent": root.findtext("cap:sent", namespaces=NS),
        "status": root.findtext("cap:status", namespaces=NS),
        "msgType": root.findtext("cap:msgType", namespaces=NS),
    }
    for tag in ["event", "urgency", "severity", "certainty", "headline"]:
        row[tag] = info.findtext(f"cap:{tag}", namespaces=NS)
    for param in info.findall("cap:parameter", NS):
        row[param.findtext("cap:valueName", namespaces=NS)] = param.findtext("cap:value", namespaces=NS)
    return row

def read_center(center):
    """Read the reference Atom snapshot and its CAP messages."""
    stamp = SECOND
    atom_file = SAMPLES / f"{center}_{stamp}.xml"
    root = ET.parse(atom_file).getroot()
    rows = []
    for number, entry in enumerate(root.findall("a:entry", NS), start=1):
        row = {
            "center": center,
            "snapshot": stamp,
            "entry_id": entry.findtext("a:id", namespaces=NS),
            "title": entry.findtext("a:title", namespaces=NS),
            "updated": entry.findtext("a:updated", namespaces=NS),
            "lat": entry.findtext("geo:lat", namespaces=NS),
            "lon": entry.findtext("geo:long", namespaces=NS),
        }
        cap_file = SAMPLES / f"{center}_cap{number}_{stamp}.xml"
        if cap_file.exists():
            row.update({f"cap_{k}": v for k, v in parse_cap(cap_file).items()})
        rows.append(row)
    print(f"{center}: {atom_file.name} -> {len(rows)} entries")
    return pd.DataFrame(rows)

tsunami_gov = pd.concat([read_center("ntwc"), read_center("ptwc")], ignore_index=True)
tsunami_gov.T


ntwc: ntwc_20261107_140301.xml -> 1 entries
ptwc: ptwc_20261107_140301.xml -> 1 entries


,0,1
center,ntwc,ptwc
snapshot,20261107_140301,20261107_140301
entry_id,urn:uuid:6d7fa756-1e86-4fe3-841a-e10e7ebb7430,urn:uuid:ba64bc82-07f7-45fb-ab6c-25420fca6851
title,"75 miles SW of Eureka, California",BONIN ISLANDS JAPAN REGION
updated,2026-11-05T05:01:47Z,2026-10-25T02:27:01Z
lat,40.396,28.
lon,-125.484,139.9
cap_identifier,PAAQ-1-to0j1g,PHEB-1-26297050
cap_sent,2026-11-05T05:01:47-00:00,2026-10-25T02:27:01-00:00
cap_status,Actual,Actual


## 3. Source Profiling

For each source, show record and column counts, field types, null percentages, duplicate IDs, and numeric ranges. The workshop requires at least 200 records per source, or the complete feed when fewer exist.

In [6]:

def profile(df, name, id_col):
    print(f"{name}: {len(df)} rows x {df.shape[1]} columns")
    print(f"  duplicated {id_col}: {df[id_col].duplicated().sum()}")
    print("  200-record requirement:", "yes" if len(df) >= 200 else "full feed required when fewer than 200")
    table = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "null_count": df.isna().sum(),
        "null_%": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique(),
    })
    numeric = df.select_dtypes("number")
    table["min"] = numeric.min()
    table["max"] = numeric.max()
    return table.sort_values("null_%", ascending=False)


### 3.1 NOAA HazEL events

In [7]:
profile(noaa, 'NOAA HazEL tsunami events', 'id')

NOAA HazEL tsunami events: 221 rows x 56 columns
  duplicated id: 0
  200-record requirement: yes


,dtype,null_count,null_%,n_unique,min,max
missingAmountOrder,float64,220,99.5,1,3.000,3.000
tsMtIi,float64,220,99.5,1,8.400,8.400
missing,float64,220,99.5,1,667.000,667.000
damageMillionsDollars,float64,217,98.2,4,118.648,220136.600
housesDamaged,float64,216,97.7,5,3.000,280920.000
missingTotal,float64,215,97.3,6,1.000,667.000
missingAmountOrderTotal,float64,215,97.3,3,1.000,3.000
housesDamagedAmountOrder,float64,214,96.8,3,1.000,4.000
housesDestroyed,float64,213,96.4,8,1.000,123661.000
housesDestroyedAmountOrder,float64,211,95.5,3,1.000,4.000


### 3.2 USGS FDSN catalog

In [8]:
profile(usgs_hist, 'USGS FDSN catalog', 'id')

USGS FDSN catalog: 343 rows x 24 columns
  duplicated id: 0
  200-record requirement: yes


,dtype,null_count,null_%,n_unique,min,max
magError,float64,343,100.0,0,NaN,NaN
dmin,float64,341,99.4,2,0.3225,0.5137
horizontalError,float64,341,99.4,2,0.5500,1.8100
magNst,float64,298,86.9,43,0.0000,299.0000
depthError,float64,277,80.8,49,1.3100,31.6100
rms,float64,7,2.0,80,0.1100,1.5000
time,str,0,0.0,343,NaN,NaN
latitude,float64,0,0.0,341,-57.6640,60.4780
gap,float64,0,0.0,234,9.5000,239.0000
nst,int64,0,0.0,256,10.0000,832.0000


### 3.3 USGS real-time feed

In [9]:
profile(usgs_rt, 'USGS real-time feed', 'id')

USGS real-time feed: 726 rows x 31 columns
  duplicated id: 0
  200-record requirement: yes


,dtype,null_count,null_%,n_unique,min,max
tz,object,726,100.0,0,NaN,NaN
alert,str,652,89.8,3,NaN,NaN
mmi,float64,641,88.3,65,0.000000e+00,8.750000e+00
felt,float64,598,82.4,33,0.000000e+00,2.018000e+03
cdi,float64,598,82.4,28,0.000000e+00,7.900000e+00
mag,float64,0,0.0,29,4.500000e+00,7.700000e+00
id,str,0,0.0,726,NaN,NaN
url,str,0,0.0,726,NaN,NaN
updated,int64,0,0.0,725,1.792143e+12,1.794060e+12
time,int64,0,0.0,726,1.791469e+12,1.794051e+12


### 3.4 tsunami.gov

In [10]:

print("Entries per center (each Atom feed only keeps the latest message):")
print(tsunami_gov.groupby("center").size())
profile(tsunami_gov, "tsunami.gov (NTWC + PTWC)", "entry_id")


Entries per center (each Atom feed only keeps the latest message):
center
ntwc    1
ptwc    1
dtype: int64
tsunami.gov (NTWC + PTWC): 2 rows x 26 columns
  duplicated entry_id: 0
  200-record requirement: full feed required when fewer than 200


,dtype,null_count,null_%,n_unique,min,max
cap_WEAK53,str,1,50.0,1,NaN,NaN
cap_ProductDefinition,str,1,50.0,1,NaN,NaN
cap_NWSUGC,str,1,50.0,1,NaN,NaN
entry_id,str,0,0.0,2,NaN,NaN
updated,str,0,0.0,2,NaN,NaN
lat,str,0,0.0,2,NaN,NaN
lon,str,0,0.0,2,NaN,NaN
title,str,0,0.0,2,NaN,NaN
center,str,0,0.0,2,NaN,NaN
snapshot,str,0,0.0,1,NaN,NaN


### 3.5 Coordinates, timestamps, and units

### 3.5.1 Coordinate validity (WGS84: latitude −90..90, longitude −180..180)

In [11]:

def check_coords(df, lat, lon, name):
    lat_v = pd.to_numeric(df[lat], errors="coerce")
    lon_v = pd.to_numeric(df[lon], errors="coerce")
    return {
        "source": name,
        "rows": len(df),
        "missing": int((lat_v.isna() | lon_v.isna()).sum()),
        "lat_out_of_range": int(((lat_v < -90) | (lat_v > 90)).sum()),
        "lon_out_of_range": int(((lon_v < -180) | (lon_v > 180)).sum()),
        "at_0_0": int(((lat_v == 0) & (lon_v == 0)).sum()),
    }

pd.DataFrame([
    check_coords(noaa, "latitude", "longitude", "NOAA HazEL"),
    check_coords(usgs_hist, "latitude", "longitude", "USGS FDSN"),
    check_coords(usgs_rt, "lat", "lon", "USGS real-time"),
    check_coords(tsunami_gov, "lat", "lon", "tsunami.gov"),
])


,source,rows,missing,lat_out_of_range,lon_out_of_range,at_0_0
0,NOAA HazEL,221,9,0,0,0
1,USGS FDSN,343,0,0,0,0
2,USGS real-time,726,0,0,0,0
3,tsunami.gov,2,0,0,0,0


### 3.5.2 Timestamp formats and time zones

In [12]:

print("NOAA date columns (year, month, day, hour, minute, second), example:")
print(noaa.loc[noaa["time_complete"], ["year", "month", "day", "hour", "minute", "second"]].head(2).to_string(index=False))
print("USGS CSV time (ISO 8601, UTC):", usgs_hist["time"].iloc[0])
print("USGS real-time time (epoch milliseconds, UTC):", usgs_rt["time"].iloc[0], "->", usgs_rt["time_utc"].iloc[0])
print("tsunami.gov Atom updated (ISO 8601, UTC):", tsunami_gov["updated"].iloc[0])
if "cap_EventOriginTime" in tsunami_gov:
    print("tsunami.gov CAP EventOriginTime:", tsunami_gov["cap_EventOriginTime"].dropna().iloc[0])

ranges = pd.DataFrame({
    "first": [noaa["time_utc"].min(), usgs_hist["time_utc"].min(), usgs_rt["time_utc"].min()],
    "last": [noaa["time_utc"].max(), usgs_hist["time_utc"].max(), usgs_rt["time_utc"].max()],
}, index=["NOAA HazEL", "USGS FDSN", "USGS real-time"])
print("Time coverage of each sample:")
ranges


NOAA date columns (year, month, day, hour, minute, second), example:
 year  month  day  hour  minute  second
 2010      1    3  21.0    48.0     5.3
 2010      1    3  22.0    36.0    27.9
USGS CSV time (ISO 8601, UTC): 2011-12-26T04:48:08.870Z
USGS real-time time (epoch milliseconds, UTC): 1794051338338 -> 2026-11-07 11:35:38.338000+00:00
tsunami.gov Atom updated (ISO 8601, UTC): 2026-11-05T05:01:47Z
tsunami.gov CAP EventOriginTime: 2026-11-05T04:56:04-00:00
Time coverage of each sample:


,first,last
NOAA HazEL,2010-01-02 00:00:00.000+00:00,2025-12-12 02:44:11.000+00:00
USGS FDSN,2010-01-02 08:45:33.020+00:00,2011-12-26 04:48:08.870+00:00
USGS real-time,2026-10-08 14:11:53.009+00:00,2026-11-07 11:35:38.338+00:00


### 3.5.3 Units and severity scales

In [13]:

print("USGS FDSN magnitude types:")
print(usgs_hist["magType"].value_counts().to_string())
print("USGS real-time magnitude types:")
print(usgs_rt["magType"].value_counts().to_string())

units = pd.DataFrame([
    ["NOAA HazEL", "eqMagnitude", "magnitude (type not given)", "eqDepth: km (verify in HazEL docs)"],
    ["USGS FDSN / real-time", "mag + magType", "mww, mb, mwc, ms, ml ...", "depth: km"],
    ["tsunami.gov Atom/CAP", "Preliminary Magnitude / EventPreliminaryMagnitude", "Ml, Mw ... (preliminary)", "CAP EventDepth: km; text bulletin: miles"],
], columns=["source", "severity field", "magnitude type", "depth unit"])
units


USGS FDSN magnitude types:
magType
mww    138
mwc    126
mb      42
mwb     34
mw       2
ms       1
USGS real-time magnitude types:
magType
mb     609
mww    104
mwr      8
mw       2
ml       2
mwb      1


,source,severity field,magnitude type,depth unit
0,NOAA HazEL,eqMagnitude,magnitude (type not given),eqDepth: km (verify in HazEL docs)
1,USGS FDSN / real-time,mag + magType,"mww, mb, mwc, ms, ml ...",depth: km
2,tsunami.gov Atom/CAP,Preliminary Magnitude / EventPreliminaryMagnitude,"Ml, Mw ... (preliminary)",CAP EventDepth: km; text bulletin: miles


## 4. Geographic Scope

The polygon crosses the antimeridian and stores longitudes from about −255 to −66. Only the point-in-polygon test shifts positive event longitudes by 360 degrees. Source coordinates remain in WGS84. Counts below include records with missing coordinates in the before total.

In [14]:
polygon = json.loads(POLYGON.read_text(encoding="utf-8"))
ring = polygon["features"][0]["geometry"]["coordinates"][0]

def in_polygon(lon, lat):
    """Test a point against the polygon, shifting positive longitude across the date line."""
    if pd.isna(lon) or pd.isna(lat):
        return False
    lon, lat = float(lon), float(lat)
    x = lon - 360 if lon > 0 else lon
    inside = False
    for (x1, y1), (x2, y2) in zip(ring, ring[1:]):
        if (y1 > lat) != (y2 > lat) and x < (x2 - x1) * (lat - y1) / (y2 - y1) + x1:
            inside = not inside
    return inside

# Keep WGS84 coordinates in the source tables; transform longitude only inside the test.
for source, frame, lon, lat in [
    ("NOAA HazEL", noaa, "longitude", "latitude"),
    ("USGS FDSN", usgs_hist, "longitude", "latitude"),
    ("USGS real-time", usgs_rt, "lon", "lat"),
    ("tsunami.gov", tsunami_gov, "lon", "lat"),
]:
    frame["in_pacific"] = [in_polygon(x, y) for x, y in zip(frame[lon], frame[lat])]
    missing = int((frame[lon].isna() | frame[lat].isna()).sum())
    outside = len(frame) - int(frame["in_pacific"].sum()) - missing
    print(f"{source}: {len(frame)} total -> {int(frame['in_pacific'].sum())} inside, "
          f"{outside} outside, {missing} without coordinates")

noaa_pacific = noaa.loc[noaa["in_pacific"]].copy()
usgs_hist_pacific = usgs_hist.loc[usgs_hist["in_pacific"]].copy()
usgs_rt_pacific = usgs_rt.loc[usgs_rt["in_pacific"]].copy()
tsunami_gov_pacific = tsunami_gov.loc[tsunami_gov["in_pacific"]].copy()

NOAA HazEL: 221 total -> 205 inside, 7 outside, 9 without coordinates
USGS FDSN: 343 total -> 325 inside, 18 outside, 0 without coordinates
USGS real-time: 726 total -> 544 inside, 182 outside, 0 without coordinates
tsunami.gov: 2 total -> 2 inside, 0 outside, 0 without coordinates


## 5. Incremental Source Comparison

Compare only the two saved snapshots from `20261107_050434` and `20261107_140301`. The USGS feed is a rolling 30-day window.

In [15]:
from datetime import datetime, timezone

def load_feed(path):
    """Return the feed metadata and its events indexed by id."""
    feed = json.loads(path.read_text(encoding="utf-8"))
    return feed["metadata"], {f["id"]: f for f in feed["features"]}

def utc(ms):
    return datetime.fromtimestamp(ms / 1000, timezone.utc)

first_file = SAMPLES / f"usgs_realtime_{FIRST}.geojson"
second_file = SAMPLES / f"usgs_realtime_{SECOND}.geojson"
meta1, ev1 = load_feed(first_file)
meta2, ev2 = load_feed(second_file)

new_ids = ev2.keys() - ev1.keys()
removed_ids = ev1.keys() - ev2.keys()
modified_ids = [i for i in ev1.keys() & ev2.keys()
                if ev1[i]["properties"] != ev2[i]["properties"] or ev1[i]["geometry"] != ev2[i]["geometry"]]

print("Snapshots:", first_file.name, "->", second_file.name)
print("Records:", len(ev1), "->", len(ev2))
print("New:", len(new_ids), "| Modified:", len(modified_ids), "| Removed:", len(removed_ids))

Snapshots: usgs_realtime_20261107_050434.geojson -> usgs_realtime_20261107_140301.geojson
Records: 714 -> 726
New: 22 | Modified: 25 | Removed: 10


### 5.1 Removed events: deleted or expired?

The feed is a rolling 30-day window, so an event leaves it when its origin time becomes older than 30 days.

In [16]:
cutoff_ms = meta2["generated"] - 30 * 24 * 3600 * 1000
removed_times = sorted(ev1[i]["properties"]["time"] for i in removed_ids)
expired = [t for t in removed_times if t < cutoff_ms]

print("Second snapshot generated:", utc(meta2["generated"]), "| 30-day cutoff:", utc(cutoff_ms))
print(f"Removed events older than the cutoff: {len(expired)} of {len(removed_ids)}")
print("Origin times of the removed events:", utc(removed_times[0]), "->", utc(removed_times[-1]))
print("Oldest event kept in the second snapshot:", utc(min(e["properties"]["time"] for e in ev2.values())))

Second snapshot generated: 2026-11-07 14:03:01+00:00 | 30-day cutoff: 2026-10-08 14:03:01+00:00
Removed events older than the cutoff: 10 of 10
Origin times of the removed events: 2026-10-08 05:06:50.387000+00:00 -> 2026-10-08 11:36:34.860000+00:00
Oldest event kept in the second snapshot: 2026-10-08 14:11:53.009000+00:00


### 5.2 Modified events: what changed?

The comparison includes `geometry` (longitude, latitude, depth), not only `properties`.

In [17]:
field_changes = {}
shifts_km, depth_changes, mag_changes, magtype_changes, only_meta = [], [], [], [], []
for i in modified_ids:
    p1, p2 = ev1[i]["properties"], ev2[i]["properties"]
    changed = {f for f in p1.keys() | p2.keys() if p1.get(f) != p2.get(f)}
    for f in changed:
        field_changes[f] = field_changes.get(f, 0) + 1
    (lon1, lat1, d1), (lon2, lat2, d2) = ev1[i]["geometry"]["coordinates"], ev2[i]["geometry"]["coordinates"]
    if (lon1, lat1, d1) != (lon2, lat2, d2):
        field_changes["geometry"] = field_changes.get("geometry", 0) + 1
        # Equirectangular approximation, accurate enough for shifts of a few kilometers
        dx = (lon2 - lon1) * 111.32 * math.cos(math.radians(lat1))
        dy = (lat2 - lat1) * 110.57
        shifts_km.append(math.hypot(dx, dy))
        depth_changes.append(abs(d2 - d1))
    elif changed <= {"updated", "felt"}:
        only_meta.append(i)
    if p1["mag"] != p2["mag"]:
        mag_changes.append((i, p1["mag"], p2["mag"], p2["magType"]))
    if p1["magType"] != p2["magType"]:
        magtype_changes.append((i, p1["magType"], p2["magType"]))

shifts_km.sort()
print("Events modified:", len(modified_ids))
print("Fields changed (number of events):", dict(sorted(field_changes.items(), key=lambda kv: -kv[1])))
print(f"Relocated: {len(shifts_km)} | epicenter shift {shifts_km[0]:.1f}-{shifts_km[-1]:.1f} km "
      f"(median {shifts_km[len(shifts_km) // 2]:.1f} km) | largest depth change {max(depth_changes):.1f} km")
print("Magnitude changes (id, before, after, type):", mag_changes)
print("Magnitude-type changes:", magtype_changes or "none")
print("Only 'updated' or 'felt' changed:", len(only_meta))

Events modified: 25
Fields changed (number of events): {'updated': 25, 'dmin': 21, 'nst': 21, 'time': 21, 'rms': 21, 'geometry': 21, 'title': 20, 'place': 18, 'gap': 7, 'sig': 6, 'mag': 6, 'felt': 2, 'types': 1, 'mmi': 1}
Relocated: 21 | epicenter shift 1.5-29.1 km (median 5.9 km) | largest depth change 48.9 km
Magnitude changes (id, before, after, type): [('us6000u35l', 4.5, 4.6, 'mb'), ('us6000u4b1', 4.6, 4.5, 'mb'), ('us6000u4sv', 4.5, 4.6, 'mb'), ('us6000u4m5', 4.9, 5.1, 'mb'), ('us6000u3ch', 4.8, 4.7, 'mb'), ('us6000u4m6', 4.9, 5, 'mb')]
Magnitude-type changes: none
Only 'updated' or 'felt' changed: 4


In [18]:
# How late do revisions arrive? Days between each modified event's origin time and its new `updated`
ages = sorted((ev2[i]["properties"]["updated"] - ev2[i]["properties"]["time"]) / 86_400_000 for i in modified_ids)
print(f"Days from origin to revision: min {ages[0]:.1f}, median {ages[len(ages) // 2]:.1f}, max {ages[-1]:.1f}")

Days from origin to revision: min 1.0, median 18.3, max 28.8


### 5.3 New events

New events are checked against the Pacific polygon of `config/pacific_polygon.geojson` (longitudes stored from −255.5 to −66 because it crosses the antimeridian).

In [19]:
new_mags = sorted(ev2[i]["properties"]["mag"] for i in new_ids)
new_inside = sum(in_polygon(*ev2[i]["geometry"]["coordinates"][:2]) for i in new_ids)
print(f"New events: {len(new_ids)} | magnitude {new_mags[0]}-{new_mags[-1]} | "
      f"M >= 6: {sum(m >= 6 for m in new_mags)} | inside the Pacific polygon: {new_inside}")

New events: 22 | magnitude 4.5-5.5 | M >= 6: 0 | inside the Pacific polygon: 19


### 5.4 tsunami.gov feeds

In [20]:
ATOM = "{http://www.w3.org/2005/Atom}"
for center in ["ntwc", "ptwc"]:
    snapshots = []
    for stamp in [FIRST, SECOND]:
        path = SAMPLES / f"{center}_{stamp}.xml"
        entries = ET.parse(path).getroot().findall(f"{ATOM}entry")
        snapshots.append({(e.findtext(f"{ATOM}id"), e.findtext(f"{ATOM}updated")) for e in entries})
    first_entries, second_entries = snapshots
    print(f"{center.upper()}: {len(first_entries)} -> {len(second_entries)} entries | "
          f"new {len(second_entries - first_entries)}, removed {len(first_entries - second_entries)}")

NTWC: 1 -> 1 entries | new 0, removed 0
PTWC: 1 -> 1 entries | new 0, removed 0


## 6. Data Quality Problems

Each row gives a measured issue and a real example identifier.

In [21]:

problems = []

def add(problem, source, evidence, example):
    problems.append({"problem": problem, "source": source, "evidence": evidence, "example": example})

# 1. NOAA: official warning status missing for most events
missing = noaa["warningStatusId"].isna()
add("Official warning status missing", "NOAA HazEL",
    f"{missing.sum()} of {len(noaa)} events ({missing.mean():.0%}) have no warningStatusId",
    f"event id {noaa.loc[missing, 'id'].iloc[0]}")

# 2. NOAA: doubtful / erroneous events
bad = noaa[noaa["eventValidity"] <= 2]
add("Erroneous or doubtful tsunami events", "NOAA HazEL",
    f"{len(bad)} events with validity <= 2: {bad['validity'].value_counts().to_dict()}",
    f"event id {bad['id'].iloc[0]} -> {bad['validity'].iloc[0]}")

# 3. NOAA: incomplete time of day
no_time = ~noaa["time_complete"]
add("Time of day missing", "NOAA HazEL",
    f"{no_time.sum()} events have no hour/minute (only the date is known)",
    f"event id {noaa.loc[no_time, 'id'].iloc[0]} on {noaa.loc[no_time, 'time_utc'].iloc[0].date()}")

# 4. NOAA: no coordinates
no_coord = noaa["latitude"].isna() | noaa["longitude"].isna()
add("Events without coordinates", "NOAA HazEL",
    f"{no_coord.sum()} events cannot be placed on a map",
    f"event id {noaa.loc[no_coord, 'id'].iloc[0]} ({noaa.loc[no_coord, 'locationName'].iloc[0]})")

# 5. NOAA: tsunamis that were not caused by an earthquake
non_eq = ~noaa["causeCode"].isin([1, 3, 4, 5])
add("Tsunamis without an earthquake cause", "NOAA HazEL",
    f"{non_eq.sum()} events: {noaa.loc[non_eq, 'cause'].value_counts().to_dict()}",
    f"event id {noaa.loc[non_eq, 'id'].iloc[0]} -> {noaa.loc[non_eq, 'cause'].iloc[0]}")

# 6. USGS CSV: columns that are completely empty
empty_cols = [c for c in usgs_hist.columns if usgs_hist[c].isna().all()]
add("Columns that are always empty", "USGS FDSN (CSV)",
    f"{len(empty_cols)} columns are 100% null: {empty_cols}",
    f"event id {usgs_hist['id'].iloc[0]}")

# 7. USGS: identifiers with different formats
odd = usgs_hist[usgs_hist["net"] != "us"]
if len(odd):
    add("Identifiers from different networks", "USGS FDSN (CSV)",
        f"net values: {usgs_hist['net'].value_counts().to_dict()}",
        f"id {odd['id'].iloc[0]} (net={odd['net'].iloc[0]}), {odd['place'].iloc[0]}")

# 8. USGS real-time: the tsunami flag does not mean a tsunami
flagged = usgs_rt[usgs_rt["tsunami"] == 1]
add("`tsunami` flag is only a bulletin indicator", "USGS real-time",
    f"{len(flagged)} of {len(usgs_rt)} events have tsunami=1; the flag does not mean a tsunami happened",
    f"id {flagged['id'].iloc[0]} ({flagged['place'].iloc[0]})" if len(flagged) else "none in this snapshot")

# 9. tsunami.gov: location as free text, depth in different units
atom_text = tsunami_gov[["center", "title"]].head(2).to_dict("records")
add("Location as free text; units mixed", "tsunami.gov",
    "location is free text; the bulletin uses miles and CAP depth uses kilometers",
    f"CAP {tsunami_gov['cap_identifier'].iloc[0]}: {atom_text[0]['title']}")

# The PTWC CAP origin date is one day earlier than the matching USGS event.
ptwc = tsunami_gov.loc[tsunami_gov["center"] == "ptwc"].iloc[0]
ptwc_event = usgs_rt.loc[usgs_rt["id"] == "us6000u4r9"].iloc[0]
cap_origin = pd.to_datetime(ptwc["cap_EventOriginTime"], utc=True)
origin_gap = ptwc_event["time_utc"] - cap_origin
add("CAP origin date differs from USGS", "tsunami.gov",
    f"CAP and USGS origin times differ by {origin_gap.total_seconds() / 3600:.2f} hours",
    f"CAP {ptwc['cap_identifier']} / USGS {ptwc_event['id']}")
pd.DataFrame(problems)

,problem,source,evidence,example
0,Official warning status missing,NOAA HazEL,169 of 221 events (76%) have no warningStatusId,event id 4522
1,Erroneous or doubtful tsunami events,NOAA HazEL,"9 events with validity <= 2: {'Questionable Tsunami': 8, 'Erroneous entry': 1}",event id 5407 -> Questionable Tsunami
2,Time of day missing,NOAA HazEL,23 events have no hour/minute (only the date is known),event id 4762 on 2010-04-13
3,Events without coordinates,NOAA HazEL,9 events cannot be placed on a map,event id 5739 (CHEJU (JEJU) STRAIT)
4,Tsunamis without an earthquake cause,NOAA HazEL,"36 events: {'Landslide': 17, 'Meteorological': 9, 'Volcano': 6, 'Unknown': 2...",event id 4762 -> Landslide
5,Columns that are always empty,USGS FDSN (CSV),1 columns are 100% null: ['magError'],event id usp000jcna
6,Identifiers from different networks,USGS FDSN (CSV),"net values: {'us': 339, 'official': 2, 'ci': 1, 'nc': 1}","id official20110311054624120_30 (net=official), 2011 Great Tohoku Earthquake..."
7,`tsunami` flag is only a bulletin indicator,USGS real-time,6 of 726 events have tsunami=1; the flag does not mean a tsunami happened,"id us6000u4r9 (Bonin Islands, Japan region)"
8,Location as free text; units mixed,tsunami.gov,location is free text; the bulletin uses miles and CAP depth uses kilometers,"CAP PAAQ-1-to0j1g: 75 miles SW of Eureka, California"
9,CAP origin date differs from USGS,tsunami.gov,CAP and USGS origin times differ by 24.01 hours,CAP PHEB-1-26297050 / USGS us6000u4r9


## 7. Cross-Source Overlap

Candidate rule: origin times within 15 minutes and epicenters within 100 km. When NOAA has no time of day, compare the calendar date and mark the candidate `date_only`. These are exploratory links, not confirmed event identities.

In [22]:

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))

def norm_lon(lon):
    """USGS can return longitudes above 180 in a Pacific query; bring them to -180..180."""
    return ((lon + 180) % 360) - 180

def find_pairs(noaa_df, usgs_df, max_minutes=15, max_km=100):
    rows = []
    for _, n in noaa_df.dropna(subset=["latitude", "longitude"]).iterrows():
        if n["time_complete"]:
            close = usgs_df[(usgs_df["time_utc"] - n["time_utc"]).abs() <= pd.Timedelta(minutes=max_minutes)]
        else:
            close = usgs_df[usgs_df["time_utc"].dt.date == n["time_utc"].date()]
        best = None
        for _, u in close.iterrows():
            dist = haversine_km(n["latitude"], n["longitude"], u["latitude"], norm_lon(u["longitude"]))
            if dist <= max_km and (best is None or dist < best["distance_km"]):
                best = {
                    "noaa_id": n["id"], "usgs_id": u["id"],
                    "noaa_time": n["time_utc"], "usgs_time": u["time_utc"],
                    "dt_seconds": round((u["time_utc"] - n["time_utc"]).total_seconds(), 2),
                    "distance_km": round(dist, 1),
                    "noaa_mag": n["eqMagnitude"], "usgs_mag": u["mag"], "usgs_magType": u["magType"],
                    "noaa_depth": n["eqDepth"], "usgs_depth": u["depth"],
                    "date_only": not n["time_complete"],
                    "place": u["place"],
                }
        if best:
            rows.append(best)
    return pd.DataFrame(rows)

# Compare only NOAA events inside the period covered by the USGS sample
start = usgs_hist["time_utc"].min().normalize()
end = usgs_hist["time_utc"].max().normalize() + pd.Timedelta(days=1)
noaa_period = noaa_pacific[
    (noaa_pacific["time_utc"] >= start) & (noaa_pacific["time_utc"] < end) & noaa_pacific["causeCode"].isin([1, 3, 4, 5])
]
print(f"Period compared: {start.date()} -> {end.date()}")
print(f"NOAA earthquake-caused tsunamis in that period: {len(noaa_period)}")

pairs = find_pairs(noaa_period, usgs_hist_pacific)
print(f"Pairs found (NOAA -> USGS): {len(pairs)}")
pairs


Period compared: 2010-01-02 -> 2011-12-27
NOAA earthquake-caused tsunamis in that period: 28
Pairs found (NOAA -> USGS): 28


,noaa_id,usgs_id,noaa_time,usgs_time,dt_seconds,distance_km,noaa_mag,usgs_mag,usgs_magType,noaa_depth,usgs_depth,date_only,place
0,4522,usp000h5nd,2010-01-03 21:48:05.300+00:00,2010-01-03 21:48:02.870+00:00,-2.43,2.2,6.6,6.6,mwc,26.0,10.0,False,"98 km SE of Gizo, Solomon Islands"
1,4523,usp000h5np,2010-01-03 22:36:27.900+00:00,2010-01-03 22:36:25.640+00:00,-2.26,2.0,7.1,7.1,mwc,25.0,10.0,False,"94 km SE of Gizo, Solomon Islands"
2,4542,usp000h5rn,2010-01-05 12:15:32.200+00:00,2010-01-05 12:15:32.210+00:00,0.01,0.0,6.8,6.8,mwc,15.0,15.4,False,"127 km SE of Gizo, Solomon Islands"
3,4682,official20100227063411530_30,2010-02-27 06:34:11.500+00:00,2010-02-27 06:34:11.530+00:00,0.03,0.0,8.8,8.8,mww,22.0,22.9,False,"2010 Maule, Chile Earthquake"
4,4722,usp000h94t,2010-03-11 14:39:43.900+00:00,2010-03-11 14:39:43.950+00:00,0.05,0.0,6.9,6.9,mwc,11.0,11.0,False,"61 km NW of Santa Cruz, Chile"
5,4782,usp000h7qu,2010-02-26 20:31:26.900+00:00,2010-02-26 20:31:26.970+00:00,0.07,0.0,7.0,7.0,mwc,25.0,25.0,False,"70 km SE of Katsuren-haebaru, Japan"
6,5222,usp000hhuc,2010-08-10 05:23:44.900+00:00,2010-08-10 05:23:44.980+00:00,0.08,0.0,7.2,7.3,mwc,25.0,25.0,False,"33 km NW of Port-Vila, Vanuatu"
7,5343,usp000hj2j,2010-08-13 21:19:33.000+00:00,2010-08-13 21:19:34.090+00:00,1.09,0.4,6.9,6.9,mwc,10.0,16.0,False,Mariana Islands region
8,5402,usp000hr97,2010-12-21 17:19:40.600+00:00,2010-12-21 17:19:40.660+00:00,0.06,0.0,7.4,7.4,mwc,14.0,14.0,False,"Bonin Islands, Japan region"
9,5403,usp000hrw0,2010-12-25 13:16:37.000+00:00,2010-12-25 13:16:37.000+00:00,0.00,0.0,7.3,7.3,mwc,16.0,16.0,False,"141 km W of Isangel, Vanuatu"


In [23]:

if len(pairs):
    summary = pd.DataFrame({
        "pairs": [len(pairs)],
        "median |dt| (s)": [pairs["dt_seconds"].abs().median()],
        "median distance (km)": [pairs["distance_km"].median()],
        "median |mag diff|": [(pairs["noaa_mag"] - pairs["usgs_mag"]).abs().median()],
    })
    display(summary)
    print("Pairs with the largest differences:")
    display(
        pairs.assign(mag_diff=(pairs["noaa_mag"] - pairs["usgs_mag"]).abs())
        .sort_values(["mag_diff", "distance_km"], ascending=False)
        .head(5)[["noaa_id", "usgs_id", "noaa_mag", "usgs_mag", "usgs_magType",
                  "noaa_depth", "usgs_depth", "distance_km", "dt_seconds"]]
    )


,pairs,median |dt| (s),median distance (km),median |mag diff|
0,28,0.075,0.0,0.0


Pairs with the largest differences:


,noaa_id,usgs_id,noaa_mag,usgs_mag,usgs_magType,noaa_depth,usgs_depth,distance_km,dt_seconds
24,5931,usp000hvkh,6.3,6.5,mwc,15.0,15.0,0.0,0.68
10,5406,usp000huc2,6.8,6.9,mww,28.0,26.0,15.9,0.21
20,5437,usp000j6r4,7.1,7.2,mww,32.0,32.0,0.0,0.31
21,5438,usp000j7dc,6.8,6.9,mww,32.0,32.0,0.0,0.09
6,5222,usp000hhuc,7.2,7.3,mwc,25.0,25.0,0.0,0.08


### 7.1 tsunami.gov bulletin vs USGS real-time feed

In [24]:

caps = tsunami_gov_pacific.dropna(subset=["cap_EventOriginTime"])
rows = []
for _, t in caps.iterrows():
    t_time = pd.to_datetime(t["cap_EventOriginTime"], utc=True)
    t_lat, t_lon = (float(x) for x in t["cap_EventLatLon"].split()[0].split(","))
    close = usgs_rt_pacific[(usgs_rt_pacific["time_utc"] - t_time).abs() <= pd.Timedelta(minutes=15)]
    for _, u in close.iterrows():
        dist = haversine_km(t_lat, t_lon, u["lat"], u["lon"])
        if dist <= 100:
            rows.append({
                "center": t["center"], "bulletin": t["cap_identifier"], "usgs_id": u["id"],
                "bulletin_time": t_time, "usgs_time": u["time_utc"],
                "dt_seconds": round((u["time_utc"] - t_time).total_seconds(), 1),
                "distance_km": round(dist, 1),
                "bulletin_mag": f"{t['cap_EventPreliminaryMagnitude']} {t['cap_EventPreliminaryMagnitudeType']}",
                "usgs_mag": f"{u['mag']} {u['magType']}",
                "bulletin_depth": t["cap_EventDepth"], "usgs_depth_km": round(u["depth"], 1),
                "usgs_tsunami_flag": u["tsunami"],
                "bulletin_says": t["cap_severity"] + " / " + t["cap_certainty"],
            })
bulletin_pairs = pd.DataFrame(rows)
print("Bulletin/USGS pairs found:", len(bulletin_pairs), "(a message can fall outside the 30-day window of the feed)")
bulletin_pairs.T


Bulletin/USGS pairs found: 1 (a message can fall outside the 30-day window of the feed)


,0
center,ntwc
bulletin,PAAQ-1-to0j1g
usgs_id,nc75462637
bulletin_time,2026-11-05 04:56:04+00:00
usgs_time,2026-11-05 04:56:02.670000+00:00
dt_seconds,-1.3
distance_km,8.7
bulletin_mag,4.5 Ml
usgs_mag,4.96 mw
bulletin_depth,13 kilometers


### 7.2 CAP messages without a strict match

Search within two days and 100 km to diagnose an unmatched bulletin. A result outside the 15-minute rule remains a candidate only.

In [25]:
# Check a two-day window for CAP messages without a strict match.
for _, message in caps.iterrows():
    if message["cap_identifier"] in set(bulletin_pairs.get("bulletin", [])):
        continue
    origin = pd.to_datetime(message["cap_EventOriginTime"], utc=True)
    lat, lon = (float(x) for x in message["cap_EventLatLon"].split()[0].split(","))
    nearby = usgs_rt_pacific[((usgs_rt_pacific["time_utc"] - origin).abs() <= pd.Timedelta(days=2)) & (usgs_rt_pacific["mag"] >= 6)]
    for _, event in nearby.iterrows():
        distance = haversine_km(lat, lon, event["lat"], event["lon"])
        if distance <= 100:
            hours = (event["time_utc"] - origin).total_seconds() / 3600
            print(f"Candidate only: {message['cap_identifier']} / {event['id']} | "
                  f"time difference {hours:+.2f} h | distance {distance:.1f} km | "
                  f"magnitude {message['cap_EventPreliminaryMagnitude']} vs {event['mag']}")

Candidate only: PHEB-1-26297050 / us6000u4r9 | time difference +24.01 h | distance 35.8 km | magnitude 7.0 vs 6.5


## 8. Conclusions

- All four sources can be loaded and profiled from saved samples. NOAA and USGS historical samples exceed 200 records; each tsunami.gov feed contains only one current entry.
- The Pacific polygon removes events outside the project area after the source queries. The original tables remain available, and event coordinates remain in WGS84.
- USGS events can change after publication, including magnitude and location; ingest updates using `updated`, while keeping events that expire from the rolling feed.
- Missing NOAA times and coordinates require explicit precision and null handling. Keep magnitude type and depth units with the values.
- Cross-source pairs are candidates based on time and distance. The PTWC CAP date discrepancy shows why a candidate needs review before it is treated as a confirmed match.